# 40. The Port Capacity & Expansion Timing Problem
## Tier 1: The Pen & Paper Method (Markov Decision Process Formulation)

### Goal
Model the port capacity expansion problem as a finite-horizon Markov Decision Process to optimize expansion timing while balancing static and dynamic capacity constraints under demand uncertainty.

### Key assumptions
- Demand follows probabilistic growth scenarios (conservative, baseline, aggressive)
- Expansion decisions are discrete (no expansion, minor, major, comprehensive)
- System transitions between capacity states based on demand realizations
- Discount factor of 0.92 represents 8% annual discount rate

### Approach (step-by-step)
1. **State Space Definition**: Define discrete capacity states with utilization rates
2. **Action Space**: Specify expansion alternatives and their costs
3. **Transition Probabilities**: Model demand growth scenarios
4. **Reward Function**: Combine revenue, costs, and congestion penalties
5. **Value Function**: Solve Bellman equation for optimal policy
6. **Policy Extraction**: Determine optimal expansion timing

### What to look for in the results
- Optimal expansion timing based on utilization thresholds
- Trade-off between upfront investment costs and long-term benefits
- Impact of demand uncertainty on policy decisions
- Sensitivity of optimal policy to parameter changes

### Concrete example (from the source)
Port of Meridian with current capacity of 3.2M TEU and demand of 2.8M TEU, facing three expansion alternatives:
- No expansion (maintain current capacity)
- Minor capacity upgrade (+15% capacity, $120M)
- Major berth expansion (+40% capacity, $450M)
- Comprehensive redevelopment (+85% capacity, $750M)

### Why this Tier exists vs earlier Tiers
This is Tier 1, providing the mathematical foundation for capacity expansion planning. It establishes the baseline theoretical framework that subsequent tiers will build upon with more practical computational approaches.

### Pros / Cons vs other approaches
**Pros:**
- Provides theoretically optimal solution
- Captures uncertainty explicitly through probabilities
- Clear mathematical formulation with provable properties

**Cons:**
- Computationally intensive for large state spaces
- Requires accurate probability estimates
- May oversimplify complex real-world constraints

### When to use this Tier
- Strategic planning with clear state definitions
- Problems with well-understood uncertainty
- Baseline for comparing heuristic approaches
- Academic analysis and policy development

In [1]:
# Import required libraries for MDP formulation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import pandas as pd

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully for MDP formulation")

Libraries imported successfully for MDP formulation


Libraries imported successfully for MDP formulation


Libraries imported successfully for MDP formulation


Libraries imported successfully for MDP formulation


Libraries imported successfully for MDP formulation


In [ ]:
@dataclass
class CapacityState:
    """Represents a discrete capacity state in the MDP"""
    capacity_mteu: float  # Capacity in million TEU
    utilization_rate: float  # Current utilization (0-1)
    infrastructure_age: float  # Years since last major upgrade
    
    def get_congestion_penalty(self) -> float:
        """Calculate congestion penalty based on utilization"""
        alpha = 1000  # Base penalty coefficient
        beta = 2.0    # Penalty growth rate
        threshold = 0.8  # Congestion threshold
        
        if self.utilization_rate <= threshold:
            return 0
        else:
            return alpha * np.exp(beta * (self.utilization_rate - threshold))

@dataclass
class ExpansionAction:
    """Represents an expansion action decision"""
    name: str
    capacity_increase: float  # Percentage increase
    cost_millions: float  # Investment cost in millions
    
class PortCapacityMDP:
    """Markov Decision Process for port capacity expansion planning"""
    
    def __init__(self):
        # Initialize problem parameters
        self.revenue_per_teu = 180  # $180 per TEU
        self.discount_factor = 0.92  # 8% annual discount rate
        self.maintenance_cost_rate = 0.02  # 2% of capacity annually
        
        # Define expansion actions
        self.actions = {
            0: ExpansionAction("No expansion", 0.0, 0),
            1: ExpansionAction("Minor upgrade", 0.15, 120),
            2: ExpansionAction("Major expansion", 0.40, 450),
            3: ExpansionAction("Comprehensive", 0.85, 750)
        }
        
        # Demand growth scenarios with probabilities
        self.demand_scenarios = {
            'conservative': {'growth_rate': 0.04, 'probability': 0.35},  # 3-5% growth
            'baseline': {'growth_rate': 0.07, 'probability': 0.45},      # 6-8% growth  
            'aggressive': {'growth_rate': 0.105, 'probability': 0.20}    # 9-12% growth
        }
        
        # Initialize state space
        self.states = self._initialize_states()
        self.value_function = {}
        self.optimal_policy = {}
    
    def _initialize_states(self) -> List[CapacityState]:
        """Create discrete capacity states for the MDP"""
        states = []
        
        # Current state: 3.2M TEU capacity with 2.8M TEU demand
        current_demand = 2.8
        current_capacity = 3.2
        current_utilization = current_demand / current_capacity
        
        states.append(CapacityState(
            capacity_mteu=current_capacity,
            utilization_rate=current_utilization,
            infrastructure_age=0
        ))
        
        # Future states with different capacity levels
        capacity_levels = [3.2, 3.68, 4.48, 5.92]  # After different expansion actions
        demand_levels = [2.8, 3.0, 3.5, 4.0]      # Different demand scenarios
        
        for capacity in capacity_levels:
            for demand in demand_levels:
                utilization = min(demand / capacity, 1.0)
                states.append(CapacityState(
                    capacity_mteu=capacity,
                    utilization_rate=utilization,
                    infrastructure_age=0
                ))
        
        return states
    
    def get_transition_probabilities(self, state: CapacityState, action: ExpansionAction) -> Dict[str, float]:
        """Get demand growth transition probabilities"""
        return self.demand_scenarios
    
    def calculate_reward(self, state: CapacityState, action: ExpansionAction) -> float:
        """Calculate immediate reward for state-action pair"""
        # Revenue from throughput (limited by capacity)
        throughput = min(state.utilization_rate * state.capacity_mteu, state.capacity_mteu)
        revenue = throughput * self.revenue_per_teu * 1000000  # Convert to dollars
        
        # Investment cost (one-time)
        investment_cost = action.cost_millions * 1000000
        
        # Congestion penalty
        congestion_penalty = state.get_congestion_penalty() * 365  # Annual penalty
        
        # Maintenance costs
        maintenance_cost = state.capacity_mteu * self.maintenance_cost_rate * 1000000
        
        # Net reward
        net_reward = revenue - investment_cost - congestion_penalty - maintenance_cost
        
        return net_reward / 1000000  # Return in millions
    
    def get_next_state(self, state: CapacityState, action: ExpansionAction, demand_growth: float) -> CapacityState:
        """Calculate next state given action and demand growth"""
        # New capacity after expansion
        new_capacity = state.capacity_mteu * (1 + action.capacity_increase)
        
        # New demand after growth
        current_demand = state.utilization_rate * state.capacity_mteu
        new_demand = current_demand * (1 + demand_growth)
        
        # New utilization rate
        new_utilization = min(new_demand / new_capacity, 1.0)
        
        return CapacityState(
            capacity_mteu=new_capacity,
            utilization_rate=new_utilization,
            infrastructure_age=state.infrastructure_age + 1
        )

print("MDP classes defined successfully")

In [ ]:
def solve_mdp_value_iteration(mdp: PortCapacityMDP, max_iterations: int = 1000, tolerance: float = 1e-6) -> Dict:
    """Solve the MDP using value iteration algorithm"""
    
    # Initialize value function
    for i, state in enumerate(mdp.states):
        mdp.value_function[i] = 0
    
    iteration = 0
    convergence_history = []
    
    while iteration < max_iterations:
        max_change = 0
        new_value_function = {}
        
        for i, state in enumerate(mdp.states):
            # Calculate expected value for each action
            action_values = []
            
            for action_id, action in mdp.actions.items():
                # Immediate reward
                immediate_reward = mdp.calculate_reward(state, action)
                
                # Expected future value
                expected_future_value = 0
                
                for scenario_name, scenario in mdp.get_transition_probabilities(state, action).items():
                    # Get next state
                    next_state = mdp.get_next_state(state, action, scenario['growth_rate'])
                    
                    # Find index of next state (simplified - use closest match)
                    next_state_idx = min(range(len(mdp.states)), 
                                        key=lambda j: abs(mdp.states[j].capacity_mteu - next_state.capacity_mteu))
                    
                    expected_future_value += scenario['probability'] * mdp.value_function[next_state_idx]
                
                # Total value for this action
                total_value = immediate_reward + mdp.discount_factor * expected_future_value
                action_values.append(total_value)
            
            # Update value function (maximum over actions)
            new_value_function[i] = max(action_values)
            max_change = max(max_change, abs(new_value_function[i] - mdp.value_function[i]))
        
        # Update value function
        mdp.value_function = new_value_function
        convergence_history.append(max_change)
        
        # Check convergence
        if max_change < tolerance:
            break
        
        iteration += 1
    
    # Extract optimal policy
    for i, state in enumerate(mdp.states):
        action_values = []
        
        for action_id, action in mdp.actions.items():
            immediate_reward = mdp.calculate_reward(state, action)
            expected_future_value = 0
            
            for scenario_name, scenario in mdp.get_transition_probabilities(state, action).items():
                next_state = mdp.get_next_state(state, action, scenario['growth_rate'])
                next_state_idx = min(range(len(mdp.states)), 
                                    key=lambda j: abs(mdp.states[j].capacity_mteu - next_state.capacity_mteu))
                expected_future_value += scenario['probability'] * mdp.value_function[next_state_idx]
            
            total_value = immediate_reward + mdp.discount_factor * expected_future_value
            action_values.append(total_value)
        
        # Best action
        best_action_id = np.argmax(action_values)
        mdp.optimal_policy[i] = best_action_id
    
    return {
        'iterations': iteration,
        'convergence_history': convergence_history,
        'final_values': mdp.value_function,
        'optimal_policy': mdp.optimal_policy
    }

print("Value iteration solver defined")

In [ ]:
# Create and solve the MDP
mdp = PortCapacityMDP()

# Solve using value iteration
solution_results = solve_mdp_value_iteration(mdp)

print(f"MDP solved in {solution_results['iterations']} iterations")
print(f"Final convergence: {solution_results['convergence_history'][-1]:.2e}")

# Display results for the current state (index 0)
current_state_idx = 0
current_state = mdp.states[current_state_idx]
optimal_action_id = solution_results['optimal_policy'][current_state_idx]
optimal_action = mdp.actions[optimal_action_id]
optimal_value = solution_results['final_values'][current_state_idx]

print("\n" + "="*60)
print("OPTIMAL EXPANSION DECISION FOR PORT OF MERIDIAN")
print("="*60)
print(f"Current Capacity: {current_state.capacity_mteu:.1f}M TEU")
print(f"Current Demand: {current_state.utilization_rate * current_state.capacity_mteu:.1f}M TEU")
print(f"Current Utilization: {current_state.utilization_rate:.1%}")
print(f"Congestion Penalty: ${current_state.get_congestion_penalty():.0f} per day")
print(f"\nOptimal Action: {optimal_action.name}")
print(f"Capacity Increase: {optimal_action.capacity_increase:.0%}")
print(f"Investment Cost: ${optimal_action.cost_millions}M")
print(f"Expected NPV: ${optimal_value:.1f}M")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Port Capacity Expansion MDP Analysis', fontsize=16, fontweight='bold')

# 1. Convergence History
axes[0, 0].plot(solution_results['convergence_history'])
axes[0, 0].set_title('Value Iteration Convergence')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Max Change in Value Function')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, alpha=0.3)

# 2. Action Comparison
action_names = [action.name for action in mdp.actions.values()]
action_values = []

for action_id, action in mdp.actions.items():
    immediate_reward = mdp.calculate_reward(current_state, action)
    expected_future_value = 0
    
    for scenario_name, scenario in mdp.get_transition_probabilities(current_state, action).items():
        next_state = mdp.get_next_state(current_state, action, scenario['growth_rate'])
        next_state_idx = min(range(len(mdp.states)), 
                            key=lambda j: abs(mdp.states[j].capacity_mteu - next_state.capacity_mteu))
        expected_future_value += scenario['probability'] * solution_results['final_values'][next_state_idx]
    
    total_value = immediate_reward + mdp.discount_factor * expected_future_value
    action_values.append(total_value)

colors = ['lightcoral' if i != optimal_action_id else 'lightgreen' for i in range(len(action_names))]
bars = axes[0, 1].bar(action_names, action_values, color=colors)
axes[0, 1].set_title('Action NPV Comparison')
axes[0, 1].set_ylabel('NPV ($M)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, value in zip(bars, action_values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'${value:.0f}M', ha='center', va='bottom')

# 3. Congestion Penalty Function
utilization_range = np.linspace(0.5, 1.0, 100)
congestion_penalties = [CapacityState(3.2, u, 0).get_congestion_penalty() for u in utilization_range]

axes[1, 0].plot(utilization_range, congestion_penalties, 'b-', linewidth=2)
axes[1, 0].axvline(x=current_state.utilization_rate, color='red', linestyle='--', 
                   label=f'Current ({current_state.utilization_rate:.1%})')
axes[1, 0].axvline(x=0.8, color='orange', linestyle=':', label='Congestion Threshold (80%)')
axes[1, 0].set_title('Congestion Penalty Function')
axes[1, 0].set_xlabel('Utilization Rate')
axes[1, 0].set_ylabel('Daily Penalty ($)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Demand Growth Scenarios
scenario_names = list(mdp.demand_scenarios.keys())
scenario_probs = [scenario['probability'] for scenario in mdp.demand_scenarios.values()]
scenario_growth = [scenario['growth_rate'] for scenario in mdp.demand_scenarios.values()]

ax2 = axes[1, 1]
bars = ax2.bar(scenario_names, scenario_probs, color=['skyblue', 'lightgreen', 'salmon'])
ax2.set_title('Demand Growth Scenario Probabilities')
ax2.set_ylabel('Probability')

# Add growth rate labels on bars
for bar, growth_rate in zip(bars, scenario_growth):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{growth_rate:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Visualization complete - MDP analysis results displayed")

In [ ]:
# Policy Summary and Recommendations
print("\n" + "="*80)
print("PORT CAPACITY EXPANSION MDP: POLICY SUMMARY & RECOMMENDATIONS")
print("="*80)

print("\n📊 CURRENT SITUATION ANALYSIS:")
print(f"   • Current Capacity: {current_state.capacity_mteu:.1f}M TEU")
print(f"   • Current Demand: {current_state.utilization_rate * current_state.capacity_mteu:.1f}M TEU")
print(f"   • Utilization Rate: {current_state.utilization_rate:.1%}")
print(f"   • Daily Congestion Cost: ${current_state.get_congestion_penalty():.0f}")
print(f"   • Annual Congestion Cost: ${current_state.get_congestion_penalty() * 365:,.0f}")

print("\n🎯 OPTIMAL EXPANSION RECOMMENDATION:")
print(f"   • Action: {optimal_action.name}")
print(f"   • Capacity Increase: {optimal_action.capacity_increase:.0%}")
print(f"   • Investment Required: ${optimal_action.cost_millions}M")
print(f"   • Expected NPV: ${optimal_value:.1f}M")

if optimal_action_id == 0:
    print("\n💡 STRATEGIC INSIGHT:")
    print("   • Current utilization does not warrant immediate expansion")
    print("   • Focus on operational optimization before capacity investment")
    print("   • Monitor demand growth trends and competitive capacity additions")
    print("   • Re-evaluate expansion when utilization exceeds 85-90%")
elif optimal_action_id == 1:
    print("\n💡 STRATEGIC INSIGHT:")
    print("   • Minor expansion provides good balance of cost and benefit")
    print("   • Addresses emerging capacity constraints proactively")
    print("   • Lower financial risk compared to major expansion")
    print("   • Can be followed by larger expansion if demand continues strong")
elif optimal_action_id == 2:
    print("\n💡 STRATEGIC INSIGHT:")
    print("   • Major expansion justified by current capacity constraints")
    print("   • Significant congestion costs warrant substantial investment")
    print("   • Provides long-term capacity buffer for growth")
    print("   • Consider phasing to manage financial impact")
else:
    print("\n💡 STRATEGIC INSIGHT:")
    print("   • Comprehensive redevelopment addresses severe capacity constraints")
    print("   • High utilization justifies substantial investment")
    print("   • Creates competitive advantage through superior capacity")
    print("   • Consider stakeholder impact and financing structure")

print("\n⚠️  RISK CONSIDERATIONS:")
print("   • Demand uncertainty: 35% conservative, 45% baseline, 20% aggressive growth")
print("   • Implementation risks: construction delays, cost overruns")
print("   • Market risks: competitor actions, trade pattern shifts")
print("   • Regulatory risks: environmental permits, community approval")

print("\n📈 NEXT STEPS:")
print("   • Validate demand projections with market analysis")
print("   • Conduct detailed feasibility study for recommended action")
print("   • Develop implementation timeline and financing plan")
print("   • Monitor key indicators quarterly for policy re-evaluation")
print("   • Consider phased approach if financing constraints exist")

print("\n" + "="*80)